[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/38_grpo_loss.ipynb)

# 🔴 Hard: GRPO Loss

Реализуйте учебный **Group Relative Policy Optimization (GRPO)** loss: group-wise REINFORCE objective с baseline, построенным из других ответов на тот же prompt. Вместо отдельной value model качество ответа оценивается относительно среднего внутри его группы.

> Граница упражнения: здесь реализуется только упрощённый `-A * log π` objective с нормализацией rewards. Полный GRPO pipeline обычно включает token-level ratios, clipping, KL-регуляризацию к reference policy, генерацию нескольких completions и маски токенов. Не добавляйте эти компоненты в требуемую функцию.

Given a batch of log-probabilities, scalar rewards, and group ids (one group per prompt), define the within-group normalized advantages:

$$A_i = \frac{r_i - \bar r_{g(i)}}{\text{std}_{g(i)} + \epsilon}$$

where \(\bar r_{g(i)}\) and \(\text{std}_{g(i)}\) are the mean and standard deviation of rewards in the group of example \(i\). В этом задании standard deviation считается с `unbiased=False`, то есть как population std.

## Интуиция и пример

Для одной группы rewards `[1, 2, 3]` среднее равно 2: первый ответ получает отрицательное преимущество, второй около нуля, третий положительное. Минимизация `-A*logp` повышает log-probability ответа с положительным A и понижает её для отрицательного A. Нормализация делает масштаб разных reward-функций более сопоставимым.

`group_ids` не обязаны идти подряд или быть отсортированы: все элементы с одинаковым id образуют одну независимую группу. `eps` защищает знаменатель, если rewards внутри группы одинаковы; тогда преимущества должны быть близки к нулю.

The GRPO loss is then the negative advantage-weighted log-probability:

$$\mathcal{L}_{\text{GRPO}} = -\mathbb{E}_i \big[\,\text{stop\_grad}(A_i)\, \log \pi_\theta(y_i)\big].$$

### Signature
```python
from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    """GRPO loss over a batch.

    logps: (B,) policy log-probs for each sampled response
    rewards: (B,) scalar rewards for each response
    group_ids: (B,) integers, same id = same prompt/group
    returns: scalar loss (Tensor)
    """
```

## План реализации

1. Создайте advantages той же формы, dtype и device, что rewards.
2. Для каждого уникального id выберите rewards группы, вычислите mean и `std(unbiased=False)`.
3. Запишите нормализованные значения на исходные позиции.
4. Отсоедините advantages от autograd и верните отрицательное среднее произведений с `logps`.

## Быстрые проверки

- В каждой невырожденной группе среднее A близко к 0, а population std — к 1.
- Перестановка batch вместе с `group_ids` не меняет scalar loss.
- Одинаковые rewards в группе дают почти нулевой вклад.
- Градиенты существуют только у `logps`, не у rewards.

## Частые ошибки

- Нормализовать rewards по всему batch вместо каждой группы.
- Оставить default `std(unbiased=True)`, особенно опасный для группы из одного элемента.
- Считать group id индексом непрерывного массива и ломаться на id вроде 10 или 42.
- Не detach advantages и пропустить градиент в reward path.

## Материалы для глубокого изучения

- [DeepSeekMath: Pushing the Limits of Mathematical Reasoning](https://arxiv.org/abs/2402.03300) — статья, вводящая GRPO.
- [Hugging Face TRL: GRPO Trainer](https://huggingface.co/docs/trl/grpo_trainer) — практический полный objective и варианты нормализации.
- [PyTorch std](https://docs.pytorch.org/docs/stable/generated/torch.std.html) — различие population/sample correction.

## Вопросы для самопроверки

1. Как group baseline заменяет отдельную value model в упрощённой постановке?
2. Почему `unbiased=False` важен для малых групп?
3. Какие части полного GRPO намеренно отсутствуют в этом упражнении?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

from torch import Tensor

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    pass  # compute normalized advantages per group and return -mean(adv.detach() * logps)

In [ ]:
# 🧪 Debug
logps = torch.tensor([0.0, -0.5, -1.0, -1.5])
rewards = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids = torch.tensor([0, 0, 1, 1])
print('Loss:', grpo_loss(logps, rewards, group_ids).item())

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('grpo_loss')